## Defining Agents

"LLM that controls the workflow" - means that LLm predicts tokens and those tokens can be interpreted to control the workflow (older definition)

(Newer definition) - "LLM agent runs tools in a loop to achieve a goal"

Common features:
1. Memory/persistence
2. Planning capabilities
3. "Autonomy"
4. LLM orchestration via Tools
5. Functionality via Tools

# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [11]:
# Imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
import requests

from gtts import gTTS

from io import BytesIO
from PIL import Image

In [12]:
# Initialization

load_dotenv(override=True)

google_api_key = os.getenv('GOOGLE_API_KEY')
if google_api_key:
    print(f"Gemini API Key exists and begins {google_api_key[:8]}")
else:
    print("Gemini API Key not set")
    
MODEL = "gemini-2.5-flash"

DB = "prices.db"

Gemini API Key exists and begins AIzaSyCn


In [13]:
#Connect to Gemini

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [14]:
#System message

system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [15]:
#Database Tool

def get_ticket_price(city):

    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()

        return (
            f"Ticket price to {city} is ${result[0]}" 
            if result 
            else "No price data available for this city"
        )

In [16]:
get_ticket_price("Paris")

DATABASE TOOL CALLED: Getting price for Paris


'Ticket price to Paris is $899.0'

In [17]:
#Tool Definition

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [18]:
#Image Generation

def artist(city):

    prompt = f"""
    An image representing a vacation in {city},
    showing tourist spots and everything unique about {city},
    in a vibrant pop-art style
    """

    url = f"https://image.pollinations.ai/prompt/{prompt}"

    response = requests.get(url)

    image = Image.open(BytesIO(response.content))

    return image

In [19]:
#Voice Generation

def talker(message):

    tts = gTTS(
        text=message,
        lang='en'
    )

    filename = "reply.mp3"

    tts.save(filename)

    return filename

In [20]:
#Main Chat Function

def chat(message, history):

    history = history or []

    messages = [
        {
            "role": "system",
            "content": system_message
        }
    ]

    # Previous chat history
    for h in history:

        messages.append({
            "role": h["role"],
            "content": h["content"]
        })

    # Current user message
    messages.append({
        "role": "user",
        "content": message
    })

    # First API call
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    cities = []

    # Handle tool calling
    while response.choices[0].finish_reason == "tool_calls":

        message_obj = response.choices[0].message

        responses = []

        for tool_call in message_obj.tool_calls:

            if tool_call.function.name == "get_ticket_price":

                arguments = json.loads(
                    tool_call.function.arguments
                )

                city = arguments.get("destination_city")

                cities.append(city)

                price_details = get_ticket_price(city)

                responses.append({
                    "role": "tool",
                    "content": price_details,
                    "tool_call_id": tool_call.id
                })

        messages.append(message_obj)

        messages.extend(responses)

        response = gemini.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    # Final reply
    reply = response.choices[0].message.content

    # Generate voice
    voice = talker(reply)

    # Generate image if city exists
    image = None

    if cities:
        image = artist(cities[0])

    # Update history
    history.append({
        "role": "user",
        "content": message
    })

    history.append({
        "role": "assistant",
        "content": reply
    })

    return history, voice, image

In [21]:
#Gradio UI

with gr.Blocks() as ui:

    gr.Markdown("# FlightAI Assistant")

    with gr.Row():

        chatbot = gr.Chatbot(
            height=500,
            type="messages"
        )

        image_output = gr.Image(
            height=500,
            interactive=False
        )

    with gr.Row():

        audio_output = gr.Audio(
            autoplay=True,
            type="filepath"
        )

    with gr.Row():

        msg = gr.Textbox(
            label="Chat with FlightAI"
        )

    def user_message(message, history):

        history = history or []

        return "", history + [{
            "role": "user",
            "content": message
        }]

    msg.submit(
        user_message,
        [msg, chatbot],
        [msg, chatbot]
    ).then(
        chat,
        [msg, chatbot],
        [chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for Paris
DATABASE TOOL CALLED: Getting price for new york
DATABASE TOOL CALLED: Getting price for Tokyo


## A bit more about what Gradio actually does:

1. Gradio constructs a frontend Svelte app based on our Python description of the UI
2. Gradio starts a server built upon the Starlette web framework listening on a free port that serves this React app
3. Gradio creates backend routes for our callbacks, like chat(), which calls our functions

And of course when Gradio generates the frontend app, it ensures that the the Submit button calls the right backend route.

That's it!

It's simple, and it has a result that feels magical.

# Let's go multi-modal!!

We can use DALL-E-3, the image generation model behind GPT-4o, to make us some images

Let's put this in a function called artist.

### Price alert: each time I generate an image it costs about 4 cents - don't go crazy with images!

## Let's bring this home:

1. A multi-modal AI assistant with image and audio generation
2. Tool callling with database lookup
3. A step towards an Agentic workflow


## The 3 types of Gradio UI

`gr.Interface` is for standard, simple UIs

`gr.ChatInterface` is for standard ChatBot UIs

`gr.Blocks` is for custom UIs where you control the components and the callbacks

# Exercises and Business Applications

Add in more tools - perhaps to simulate actually booking a flight. A student has done this and provided their example in the community contributions folder.

Next: take this and apply it to your business. Make a multi-modal AI assistant with tools that could carry out an activity for your work. A customer support assistant? New employee onboarding assistant? So many possibilities! Also, see the week2 end of week Exercise in the separate Notebook.

<table style="margin: 0; text-align: left;">
    <tr>
        <td>
            <h2 style="color:#090;">I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a HUGE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>